# 05 · Context-adjusted captain & voyage comparisons

Compara consumo residual, no consumo bruto. Ajusta por buque, ruta, carga, meteorología, corriente y velocidad; exige soporte comparable y CI95%.

In [1]:
from pathlib import Path
import sys, json, numpy as np, pandas as pd
ROOT=next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/'src/project8.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Abre el notebook dentro del repositorio del Proyecto 8.')
sys.path.insert(0,str(ROOT))
from src.project8 import *

df=pd.read_csv(ROOT/'data/raw/voyages_2026_synthetic.csv',parse_dates=['date'])
residuals,caps=adjusted_captains(df)
residuals.to_csv(ROOT/'data/processed/context_adjusted_voyage_residuals.csv',index=False)
caps.to_csv(ROOT/'outputs/tables/captain_adjusted_comparison.csv',index=False)
display(caps)

,captain_id,n_voyages,route_vessel_pairs,adjusted_fuel_residual_pct,ci95_low,ci95_high,matched_residual_pct,support_score,comparison_valid,interpretation
5,C06,375,79,-2.272083,-2.644073,-1.900092,-1.516514,1,True,Lower fuel than context-adjusted expectation
2,C03,415,80,-1.973531,-2.335218,-1.611844,-1.086594,1,True,Lower fuel than context-adjusted expectation
11,C12,391,80,-1.393276,-1.751127,-1.035425,-0.723574,1,True,Lower fuel than context-adjusted expectation
13,C14,430,80,-1.318661,-1.653540,-0.983783,-0.855639,1,True,Lower fuel than context-adjusted expectation
0,C01,411,80,-1.175427,-1.530996,-0.819858,-1.093483,1,True,Lower fuel than context-adjusted expectation
17,C18,407,80,-1.065533,-1.414916,-0.716150,-0.431686,1,True,Lower fuel than context-adjusted expectation
9,C10,402,79,-0.954665,-1.322216,-0.587113,-0.445517,1,True,Lower fuel than context-adjusted expectation
8,C09,379,78,-0.310221,-0.709325,0.088883,-0.241470,1,True,No robust difference
1,C02,388,77,-0.200283,-0.584292,0.183725,-0.342499,1,True,No robust difference
10,C11,393,80,-0.061899,-0.425736,0.301939,-0.232042,1,True,No robust difference


Validación cruzada adicional por estratos buque×ruta×carga×meteorología. Solo se interpreta una diferencia si existe soporte suficiente y ambos métodos apuntan en la misma dirección.

In [2]:
valid=caps[caps.comparison_valid]
print('Captains with comparable support:',len(valid),'of',len(caps))
print(pd.crosstab(valid.interpretation,columns='n'))

Captains with comparable support: 18 of 18
col_0                                          n
interpretation                                  
Higher fuel than context-adjusted expectation  6
Lower fuel than context-adjusted expectation   7
No robust difference                           5


Guardrail: esta salida sirve para estudiar prácticas operativas en un caso sintético. No es una clasificación de desempeño laboral.

In [3]:
assert caps.loc[caps.comparison_valid,'n_voyages'].min()>=120
print('Guardrail OK')

Guardrail OK
